# Generate CSV using NBA api


In [ ]:
import time
import random
from pathlib import Path

import pandas as pd
from nba_api.stats.static import players
from nba_api.stats.endpoints import leaguegamelog

In [ ]:
OUT_DIR = Path("nba_player_gamelogs_by_season")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LEAGUE_ID = "00"
SLEEP_BETWEEN_CALLS = (0.8, 1.6)  # jitter to reduce rate-limit risk


def season_strings(start_year: int, end_year: int) -> list[str]:
    """
    Build NBA Stats season strings from start_year to end_year inclusive.
    Example: 1996 -> '1996-97'
    """
    seasons = []
    for y in range(start_year, end_year + 1):
        seasons.append(f"{y}-{str((y + 1) % 100).zfill(2)}")
    return seasons


def get_active_player_ids() -> set[int]:
    active = players.get_active_players()
    return {p["id"] for p in active}


def fetch_league_player_gamelog(season: str, season_type: str) -> pd.DataFrame:
    """
    Pull ALL player game logs for a season + season_type using LeagueGameLog
    (player rows). Retries with backoff.
    """
    max_tries = 6
    last_err = None

    for attempt in range(1, max_tries + 1):
        try:
            time.sleep(random.uniform(*SLEEP_BETWEEN_CALLS))

            lg = leaguegamelog.LeagueGameLog(
                league_id=LEAGUE_ID,
                season=season,
                season_type_all_star=season_type,      # "Regular Season" or "Playoffs"
                player_or_team_abbreviation="P"        # IMPORTANT: player rows
            )
            return lg.get_data_frames()[0]

        except Exception as e:
            last_err = e
            wait = min((2 ** attempt) + random.random(), 30)
            print(f"[WARN] {season} {season_type}: attempt {attempt}/{max_tries} failed ({type(e).__name__}). "
                  f"Sleeping {wait:.1f}s...")
            time.sleep(wait)

    raise RuntimeError(f"Failed LeagueGameLog for {season} {season_type}: {last_err}")


def build_and_save_season(season: str, active_ids: set[int]) -> None:
    reg = fetch_league_player_gamelog(season, "Regular Season")
    reg = reg[reg["PLAYER_ID"].isin(active_ids)].copy()
    reg["SEASON"] = season
    reg["SEASON_TYPE"] = "Regular Season"

    po = fetch_league_player_gamelog(season, "Playoffs")
    po = po[po["PLAYER_ID"].isin(active_ids)].copy()
    po["SEASON"] = season
    po["SEASON_TYPE"] = "Playoffs"

    combined = pd.concat([reg, po], ignore_index=True)

    out_path = OUT_DIR / f"game_stats_{season}.csv"
    combined.to_csv(out_path, index=False)

    print(f"[OK] Saved {season}: {out_path} | rows={len(combined):,} "
          f"(reg={len(reg):,}, po={len(po):,})")


if __name__ == "__main__":
    # Past seasons to pull. Adjust start_year if you want fewer historical files.
    # This will create CSVs for 1996-97 ... 2024-25, excluding 2025-26.
    seasons = season_strings(2003, 2024)  # 2003-04 through 2024-25
    seasons = [s for s in seasons if s != "2025-26"]  # explicit exclusion

    active_ids = get_active_player_ids()
    print(f"Active players found: {len(active_ids)}")

    for s in seasons:
        try:
            build_and_save_season(s, active_ids)
        except Exception as e:
            print(f"[ERROR] Season {s} failed: {e}")

In [ ]:
OUT_DIR = Path("nba_player_gamelogs_by_season")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEASON = "2025-26"
LEAGUE_ID = "00"
SLEEP_BETWEEN_CALLS = (0.8, 1.6)


def get_active_player_ids() -> set[int]:
    active = players.get_active_players()
    return {p["id"] for p in active}


def fetch_league_player_gamelog(season: str, season_type: str) -> pd.DataFrame:
    max_tries = 6
    last_err = None

    for attempt in range(1, max_tries + 1):
        try:
            time.sleep(random.uniform(*SLEEP_BETWEEN_CALLS))

            lg = leaguegamelog.LeagueGameLog(
                league_id=LEAGUE_ID,
                season=season,
                season_type_all_star=season_type,      # "Regular Season" or "Playoffs"
                player_or_team_abbreviation="P"        # player rows
            )
            return lg.get_data_frames()[0]

        except Exception as e:
            last_err = e
            wait = min((2 ** attempt) + random.random(), 30)
            print(f"[WARN] {season} {season_type}: attempt {attempt}/{max_tries} failed ({type(e).__name__}). "
                  f"Sleeping {wait:.1f}s...")
            time.sleep(wait)

    raise RuntimeError(f"Failed LeagueGameLog for {season} {season_type}: {last_err}")


if __name__ == "__main__":
    active_ids = get_active_player_ids()
    print(f"Active players found: {len(active_ids)}")

    reg = fetch_league_player_gamelog(SEASON, "Regular Season")
    reg = reg[reg["PLAYER_ID"].isin(active_ids)].copy()
    reg["SEASON"] = SEASON
    reg["SEASON_TYPE"] = "Regular Season"

    po = fetch_league_player_gamelog(SEASON, "Playoffs")
    po = po[po["PLAYER_ID"].isin(active_ids)].copy()
    po["SEASON"] = SEASON
    po["SEASON_TYPE"] = "Playoffs"

    combined = pd.concat([reg, po], ignore_index=True)

    out_path = OUT_DIR / f"game_stats_{SEASON}.csv"
    combined.to_csv(out_path, index=False)

    print(f"[OK] Saved {SEASON}: {out_path} | rows={len(combined):,} "
          f"(reg={len(reg):,}, po={len(po):,})")